### Вовед во Биоинформатика

Изработила: Зорица Ников

Индекс: 216150

### Imports

In [10]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MinMaxScaler

from tqdm import tqdm

### Constants

In [3]:
possible_outputs = ["Hypertension", "Heart Disease", "Stroke", "Diabetes"]

In [4]:
classifiers = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(),
    'SVC': SVC(),
    'Neural Network': MLPClassifier(hidden_layer_sizes=[400, 400])
}

### Utility functions

In [14]:
def only_valid_dataset(df, output, features = None):
    filtered_df = df
    
    if features is not None:
        filtered_df = df[features]

    # Delete all of the rows that have the output -1
    filtered_df = filtered_df[filtered_df[output] != -1]

    # Delete all of the columns that are all -1
    filtered_df = filtered_df.loc[:, (filtered_df != -1).any(axis=0)]

    return filtered_df

In [4]:
def split_data(df, train_size=0.7, test_size=0.3, random_state=42):
    # Calculate the validation size
    valid_size = 1.0 - train_size - test_size
    
    train_df, temp_df = train_test_split(df, test_size=(1 - train_size), random_state=random_state)
    test_df, valid_df = train_test_split(temp_df, test_size=(valid_size / (test_size + valid_size)), random_state=random_state)
    
    return train_df, valid_df, test_df

In [7]:
def sklearn_classify(X_train, y_train, X_test, y_test):
    for classifier_name in classifiers.keys():
        classifier = classifiers[classifier_name]
        
        # Train the classifier
        classifier.fit(X_train, y_train)
    
        # Make predictions on the test set
        y_pred = classifier.predict(X_test)
    
        # Calculate accuracy
        accuracy = accuracy_score(y_test, y_pred)
        
        print(f"Accuracy of {classifier_name} on the test set: {accuracy * 100:.2f}%")

In [15]:
def split_in_x_and_y(train, valid, test, output):
    X_train = train.drop(columns=[output])
    y_train = train[output]

    X_val = valid.drop(columns=[output])
    y_val = valid[output]

    X_test = test.drop(columns=[output])
    y_test = test[output]

    return X_train, X_val, X_test, y_train, y_val, y_test

### Pytorch

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

In [5]:
def normalize(dataset):
    X_train, X_val, X_test, y_train, y_val, y_test = dataset
    # Scale the integer features
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test

In [26]:
def to_tensors(dataset):
    X_train, X_val, X_test, y_train, y_val, y_test = dataset

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
    y_val_tensor = torch.tensor(y_val.values, dtype=torch.long)
    y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

    return X_train_tensor, X_val_tensor, X_test_tensor, y_train_tensor, y_val_tensor, y_test_tensor

In [7]:
class ISDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feature = self.features[idx]
        label = self.labels[idx]
        return feature, label

In [8]:
def to_torch_datasets(dataset):
    X_train, X_val, X_test, y_train, y_val, y_test = dataset
    train_dataset = ISDataset(X_train, y_train)
    val_dataset = ISDataset(X_val, y_val)
    test_dataset = ISDataset(X_test, y_test)
    return train_dataset, val_dataset, test_dataset

In [34]:
class DenseNet(torch.nn.Module):
    def __init__(self, input_dims, output_dims = 2):
        super(DenseNet, self).__init__()

        self.input_dims = input_dims

        self.layers = nn.Sequential(
            nn.Linear(input_dims, 400),
            nn.ReLU(),
            nn.Linear(400, 400),
            nn.Dropout(0.3),
            nn.ReLU(),
            nn.Linear(400, output_dims)
        )

    def forward(self, x):
        return self.layers(x)

In [11]:
def train(neural_net: torch.nn.Module, dataset, epochs, batch_size, lr=0.002):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    neural_net = neural_net.to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(neural_net.parameters(), lr)

    dataset = normalize(dataset)
    dataset_tensors = to_tensors(dataset)
    train_dataset, val_dataset, test_dataset = to_torch_datasets(dataset_tensors)
    
    # Set the loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    for e in range(epochs):
        # Training phase
        neural_net.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch in tqdm(train_loader):
            x, y = batch[0].to(device), batch[1].to(device)
            
            optimizer.zero_grad()
            output = neural_net(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

            # Compute matrics
            train_loss += loss.item() * x.size(0)
            _, predicted = torch.max(output, 1)
            train_total += y.size(0)
            train_correct += (predicted == y).sum().item()

        train_loss /= train_total
        train_accuracy = train_correct / train_total

        # Validation phase
        neural_net.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for batch in val_loader:
                x, y = batch[0].to(device), batch[1].to(device)
                
                output = neural_net(x)
                loss = criterion(output, y)
                
                val_loss += loss.item() * x.size(0)
                _, predicted = torch.max(output, 1)
                val_total += y.size(0)
                val_correct += (predicted == y).sum().item()

        val_loss /= val_total
        val_accuracy = val_correct / val_total

        print(f"Epoch {e+1}/{epochs}, Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")

    # Test phase
    test_loss = 0.0
    test_correct = 0
    test_total = 0

    neural_net.eval()
    with torch.no_grad():
        for batch in test_loader:
            x, y = batch[0].to(device), batch[1].to(device)
            
            output = neural_net(x)
            loss = criterion(output, y)
            
            test_loss += loss.item() * x.size(0)
            _, predicted = torch.max(output, 1)
            test_total += y.size(0)
            test_correct += (predicted == y).sum().item()

    test_loss /= test_total
    test_accuracy = test_correct / test_total

    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")
    return neural_net


### Read data

In [12]:
df = pd.read_csv('./combined_dataset.csv')

In [11]:
def run_classification(df, output, features):
    filtered_df = only_valid_dataset(df, output, features)
    train_df, valid_df, test_df = split_data(filtered_df)
    X_train, X_val, X_test, y_train, y_val, y_test = split_in_x_and_y(train_df, valid_df, test_df, output)

    sklearn_classify(
        X_train, y_train,
        X_test, y_test
    )

Accuracy of Logistic Regression on the test set: 72.28%
Accuracy of Random Forest on the test set: 72.46%
Accuracy of SVC on the test set: 73.30%
Accuracy of Neural Network on the test set: 73.62%


### Run the models on all features

In [9]:
for output in possible_outputs:
    print(f"Predicting {output}...")
    run_classification(df, output, None)

Predicting Hypertension...


c:\Users\Nikov\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy of Logistic Regression on the test set: 74.01%
Accuracy of Random Forest on the test set: 71.90%
Accuracy of SVC on the test set: 74.20%
Accuracy of Neural Network on the test set: 74.40%
Predicting Heart Disease...


c:\Users\Nikov\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy of Logistic Regression on the test set: 91.71%
Accuracy of Random Forest on the test set: 90.47%
Accuracy of SVC on the test set: 91.83%
Accuracy of Neural Network on the test set: 91.88%
Predicting Stroke...


c:\Users\Nikov\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy of Logistic Regression on the test set: 96.48%
Accuracy of Random Forest on the test set: 95.75%
Accuracy of SVC on the test set: 96.48%
Accuracy of Neural Network on the test set: 96.48%
Predicting Diabetes...


c:\Users\Nikov\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy of Logistic Regression on the test set: 87.06%
Accuracy of Random Forest on the test set: 85.53%
Accuracy of SVC on the test set: 87.03%
Accuracy of Neural Network on the test set: 87.35%


### Run the models on a subset of features

In [10]:
output = "Hypertension"
features = [output, "Age", "BMI", "Smoking", "Sex", "High Cholesterol"]
run_classification(df, output, features)

In [ ]:
output = "Heart Disease"
features = [output, "Chest Pain Type", "Smoking", "Physical Activity", "Diabetes", "Sex"]
run_classification(df, output, features)

In [ ]:
output = "Heart Disease"
features = [output,"Diabetes", "Hypertension", "Age", "Smoking", "Chest Pain Type"]
run_classification(df, output, features)

In [ ]:
output = "Hypertension"
features = [output, "Fasting Blood Sugar", "Physical Activity", "Smoking", "Stroke", "Age"]
run_classification(df, output, features)

### Pytorch run to compare neural network performance

In [32]:
output = "Heart Disease"

In [35]:
filtered_df = only_valid_dataset(df, output)
train_df, valid_df, test_df = split_data(filtered_df, train_size=0.6, test_size=0.3)
dataset = split_in_x_and_y(train_df, valid_df, test_df, output)
dense_net = DenseNet(filtered_df.shape[1] - 1)
train(dense_net, dataset, epochs=20, batch_size=128, lr=0.00001)

100%|██████████| 1499/1499 [00:05<00:00, 291.85it/s]


Epoch 1/20, Train Loss: 0.2826, Train Accuracy: 0.9170, Val Loss: 0.2425, Val Accuracy: 0.9193


100%|██████████| 1499/1499 [00:05<00:00, 298.58it/s]


Epoch 2/20, Train Loss: 0.2370, Train Accuracy: 0.9177, Val Loss: 0.2264, Val Accuracy: 0.9201


100%|██████████| 1499/1499 [00:05<00:00, 291.38it/s]


...

100%|██████████| 1499/1499 [00:05<00:00, 299.09it/s]


Epoch 19/20, Train Loss: 0.2220, Train Accuracy: 0.9186, Val Loss: 0.2192, Val Accuracy: 0.9199


100%|██████████| 1499/1499 [00:05<00:00, 298.64it/s]


Epoch 20/20, Train Loss: 0.2220, Train Accuracy: 0.9185, Val Loss: 0.2191, Val Accuracy: 0.9196
Test Loss: 0.2228, Test Accuracy: 0.9186


DenseNet(
  (layers): Sequential(
    (0): Linear(in_features=11, out_features=2000, bias=True)
    (1): ReLU()
    (2): Linear(in_features=2000, out_features=400, bias=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): ReLU()
    (5): Linear(in_features=400, out_features=2, bias=True)
  )
)